In [ ]:
# import os
# import pandas as pd
# import numpy as np
import tensorflow as tf
import nltk
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense
from tensorflow.keras.models import Sequential

In [ ]:
# load the dataset
max_features = 5000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)
# (X_train, y_train), (X_test, y_test) = imdb.load_data()
print(f"training data shape: {X_train.shape}, training data labels: {y_train.shape}")
print(f"testing data shape: {X_test.shape}, testing data labbles: {y_test.shape}")


In [ ]:
# X_train[0]
# y_train[0]

In [ ]:
word_indexes = imdb.get_word_index()
word_indexes

In [ ]:
# max_sentence_len = max([len(x) for x in X_train])
# print(max_sentence_len)
max_sentence_len = 200

In [ ]:
X_train = sequence.pad_sequences(X_train, maxlen=max_sentence_len)
X_test = sequence.pad_sequences(X_test, maxlen=max_sentence_len)

In [ ]:
model = Sequential()
model.add(Embedding(max_features, 128, input_length=max_sentence_len))
model.add(SimpleRNN((128), activation = 'relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
EarlyStopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
EarlyStopping

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=128,
    validation_split=0.2,
    callbacks=[EarlyStopping]
)

In [ ]:
# save the file
model.save('/content/drive/MyDrive/Colab Notebooks/ML/RNN/RNN_IMDB_sentiment_model.h5')

In [ ]:
model.save('/content/drive/MyDrive/Colab Notebooks/ML/RNN/RNN_IMDB_sentiment_model.keras')

In [ ]:
reverse_word_indexes = dict([(value, key) for (key, value) in word_indexes.items()])
# reverse_word_indexes = {value: key for key, value in word_indexes.items()}
reverse_word_indexes

def decode_review(encoded_text):
  return ' '.join([reverse_word_indexes.get(i-3, '?') for i in encoded_text])

def preprocess_text(text):
  words = text.lower().split()
  encoded_text = [word_indexes.get(word ,2) + 3 for word in words]
  padded_review = sequence.pad_sequences([encoded_text] , maxlen=500)
  return padded_review


def predict_sentiment(review):

  preprocessed_input = preprocess_text(review)
  prediction = model.predict(preprocessed_input)
  sentiment = 'positive' if prediction[0][0] > 0.5 else 'negative'
  return sentiment, prediction[0][0]

example_input = "This movie very good and fantastic movie"


sentiment, score = predict_sentiment(example_input)

print(f"Review: {sentiment}")
print(f"Review Score: {score}")

In [ ]:

example_input = "This movie was absolutely incredible! The storyline was captivating, the acting was phenomenal, and the visuals were stunning."


sentiment, score = predict_sentiment(example_input)

print(f"Review: {sentiment}")
print(f"Review Score: {score}")

In [ ]:

example_input = "This movie was terrible, the acting was bad and the visuals were unsatisfactory."


sentiment, score = predict_sentiment(example_input)

print(f"Review: {sentiment}")
print(f"Review Score: {score}")